In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from netCDF4 import Dataset
import os, fnmatch
from mpl_toolkits.basemap import Basemap
import pickle

In [2]:
def find_files(directory, pattern, maxdepth=None):
    flist = []
    for root, dirs, files in os.walk(directory):
        for basename in files:
            if fnmatch.fnmatch(basename, pattern):
                filename = os.path.join(root, basename)
                filename = filename.replace('\\\\', os.sep)
                if maxdepth is None:
                    flist.append(filename)
                else:
                    if filename.count(os.sep)-directory.count(os.sep) <= maxdepth:
                        flist.append(filename)
    return flist

In [3]:
def drawing(lon, lat, data, region, data_type, day):
    fig = plt.figure(figsize=(12, 12), dpi = 300)
    
    if region == 'Kara':
        m = Basemap(width=1800000, height=1300000,
                    resolution='l', projection='aea',
                    lat_1=50, lat_2=55, lon_0=70, lat_0=74)
        s = 30
        fontsize = 10
        borders = [15, 27]
        
    elif region == 'EastSib':
        m = Basemap(width=3500000,height=1500000,
                    resolution='l',projection='aea',
                    lat_1=60, lat_2=65, lon_0=140, lat_0=76)
        s = 5
        fontsize = 8
        borders = [15, 25]
        
    m.drawcoastlines()
    m.fillcontinents(color='grey',lake_color='white')
    m.drawparallels(np.arange(-80.,80.,2.), labels=[True,False,True,False])
    m.drawmeridians(np.arange(-180.,180.,5.), labels=[False,True,False,True], latmax=90)
    m.drawmapboundary(fill_color='white')
    if data_type == 'sss':
        m.scatter(lon, lat, s=s, c=data,
                  cmap='jet', marker='o', latlon=True, vmin=4, vmax=36)
        cbar=plt.colorbar(label='Соленость', orientation='vertical', shrink=0.30)
        cbar.set_ticks([5,10,15,20,25,30,35])
        cs = m.contour(lon, lat, data, levels=borders, colors=['blue', 'red'], latlon=True)
        
    elif data_type == 'sst':
        m.scatter(lon, lat, s=s, c=data,
                  cmap='jet', marker='o', latlon=True, vmin=0, vmax=12)
        cbar=plt.colorbar(label='Температура', orientation='vertical', shrink=0.30)
        cbar.set_ticks([0,1,2,3,4,5,6,7,8,9,10,11,12])
    plt.title(f'{day}')
    font = {'size'   : fontsize}
    plt.rc('font', **font)
    ax = plt.gca()
    plt.savefig(f'/mnt/hippocamp/asavin/data/ESACCI/ESACCII_pictures_{region}/{day[:4]}/{day}.png')
    plt.show()
    plt.clf()
    plt.close('all')

In [4]:
def select_data(longitude, latitude, sss, borders):
    n, s, w, e, = borders
    mask = (latitude >= s) & (latitude <= n) & (longitude >= w) & (longitude <= e)
    
    rows_with_data = np.any(mask, axis=1)
    cols_with_data = np.any(mask, axis=0)

    row_idx = np.where(rows_with_data)[0]
    col_idx = np.where(cols_with_data)[0]

    lon = longitude[row_idx.min():row_idx.max() + 1,
                    col_idx.min():col_idx.max() + 1]
    
    lat = latitude[row_idx.min():row_idx.max() + 1,
               col_idx.min():col_idx.max() + 1]
    
    sss_ = sss[row_idx.min():row_idx.max() + 1,
            col_idx.min():col_idx.max() + 1]
    
    return lon, lat, sss_

In [5]:
def make_data(f, borders):
    data = Dataset(f, 'r')
    
    latitude = np.asarray(data['lat'])
    longitude = np.asarray(data['lon'])
    sss = np.asarray(data['sss'])[0]

    data.close()

    lon, lat, sss_ = select_data(longitude=longitude, latitude=latitude, sss=sss, borders=borders)
    # sss_ = np.where((lon < 55) | (lon > 105) | (lat > 80) | (lat < 70), np.nan, sss_)

    return lon, lat, sss_

In [6]:
def draw_files(year, start, finish, borders):
    files = find_files(f'/mnt/hippocamp/DATA/sattelite/ESACCI/v05.5/NHv5.5/7days/{year}/', '*.nc')
    files.sort()

    files = [f for f in files if f[-13:-9] >= start and f[-13:-9] <= finish]
    
    for file in files:
        lon, lat, sss = make_data(file, borders)
        drawing(lon=lon, lat=lat, data=sss, region='Kara', data_type='sss', day=file[-17:-9])

In [ ]:
draw_files(year=2011, start='0601', finish='1130', borders=[83,67,30,105])